In [1]:
import json
import random
import spacy
from spacy.util import minibatch, compounding
from spacy.training.example import Example

import pandas as pd
import os
from tqdm import tqdm
from spacy.tokens import DocBin

2024-01-24 14:13:58.488094: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


RuntimeError: module compiled against API version 0xe but this version of numpy is 0xd

In [2]:
def labelled_data_to_spacy_format(labeled_data):
    test_data_survey = []
    for text_details in labeled_data:
        text_details=text_details.replace("true","True")
        text_details=text_details.replace("false","False")
        text_details=eval(text_details)
        if text_details["answer"] == "accept":
            entities = {'entities': []}
            query = text_details['text']
            for span in text_details['spans']:
                entities['entities'].append((span['start'], span['end'], span['label']))

            test_data_survey.append((query, entities))
        else:
            #print(f"Rejecting: {text_details['text']}")
            pass
    return test_data_survey
    
    
def read_prodigy_labelled_json(file_path):
    with open (file_path, 'r') as file:  
        labeled_data=[line.strip() for line in file]
    labeled_data=list(set(labeled_data))
    labeled_data = labelled_data_to_spacy_format(labeled_data)
    return labeled_data


In [6]:
import os
# for file in os.listdir("./annotated/from_survey/certification"):
#     print('"'+'./annotated/from_survey/certification/'+file+'",')
# for file in os.listdir("./annotated/from_survey/stage1"):
#     print('"'+'./annotated/from_survey/stage1/'+file+'",')
for file in os.listdir("./annotated/from_survey/Training_Files_24_01_24"):
    print('"'+'./annotated/from_survey/Training_Files_24_01_24/'+file+'",')

"./annotated/from_survey/Training_Files_24_01_24/stacie_test_queries_1_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_test_queries_3_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_ul_queries_1_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_ul_queries_2_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse10_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse11_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse15_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse1_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse2_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse3_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse4_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse6_labelled.xlsx",
"./an

In [7]:
paths = ["./annotated/from_survey/Training_Files_24_01_24/stacie_test_queries_1_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_test_queries_3_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_ul_queries_1_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/stacie_ul_queries_2_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse10_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse11_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse15_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse1_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse2_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse3_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse4_reviewed.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse6_labelled.xlsx",
"./annotated/from_survey/Training_Files_24_01_24/SurveyResponse9_labelled.xlsx"]
labeled_data_from_survey = []
for file_path in paths:
    survey_data_df = pd.read_excel(file_path,engine='openpyxl')
    survey_data_df = survey_data_df[survey_data_df['accept']==True]
    labeled_data_from_survey.extend(survey_data_df['spacy_format'].apply(lambda x: x.strip('"')).tolist())

print(len(labeled_data_from_survey))
labeled_data_from_survey = list(set(labeled_data_from_survey))
print(len(labeled_data_from_survey))
labeled_data_from_survey = [eval(labeld_query) for labeld_query in labeled_data_from_survey]


random.shuffle(labeled_data_from_survey)

from sklearn.model_selection import train_test_split
labeled_data_from_survey, labeled_test_data = train_test_split( labeled_data_from_survey, test_size=0.15, random_state=42)
print(len(labeled_data_from_survey),len(labeled_test_data))

4271
4134
3513 621


<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


In [8]:
with open ('./annotated/survey_query_train_data.txt', 'w', encoding="utf-8") as file:  
    for line in labeled_data_from_survey: 
        file.write(str(line))
        file.write('\n')
        
with open ('./annotated/survey_query_test_data.txt', 'w', encoding="utf-8") as file:  
    for line in labeled_test_data: 
        file.write(str(line))
        file.write('\n')

with open ('./annotated/survey_query_train_data.txt', 'r',encoding='utf-8' ) as file:  
    labeled_data_from_survey=[eval(line.strip()) for line in file]

with open ('./annotated/survey_query_test_data.txt', 'r',encoding='utf-8' ) as file:  
    labeled_test_data=[eval(line.strip()) for line in file]

In [9]:
len(labeled_data_from_survey), len(labeled_test_data)

(3513, 621)

In [10]:
labeled_data_from_survey = labeled_data_from_survey*10
random.shuffle(labeled_data_from_survey)
len(labeled_data_from_survey)

35130

In [11]:
labeled_test_data= labeled_test_data*10
random.shuffle(labeled_test_data)
len(labeled_test_data)

6210

In [12]:
# import re
# for user_query in labeled_data_from_survey[0:100]:
#     lbld_query = user_query
    
#     for entiti_data in lbld_query[1]['entities']:
#         if entiti_data[2]=="MODIFIER":
#             print(lbld_query[0])
#             print(entiti_data[2], " : ", lbld_query[0][entiti_data[0]:entiti_data[1]])
#             #print(re.findall(r'(\d*\.?\d+\s*(?:-|&|and|to|~|)\s*\d*\.?\d+\s*)', lbld_query[0][entiti_data[0]:entiti_data[1]]))
#     print("\n")

In [13]:
# for user_query in labeled_data[500:600]:
#     lbld_query = user_query
#     print(lbld_query[0])
#     for entiti_data in lbld_query[1]['entities']:
#         print(entiti_data[2], " : ", lbld_query[0][entiti_data[0]:entiti_data[1]])
#     print("\n")

In [14]:
# filename1 = './annotated/user_queries_train_07_20_23.txt'

# with open (filename1, 'r',encoding="utf-8") as file:  
#     training_data1=[line.strip() for line in file]

filename2 = './annotated/user_queries_train.txt'

with open (filename2, 'r',encoding="utf-8") as file:  
    training_data2=[line.strip() for line in file]

training_data  = training_data2
training_data = list(set(training_data))
training_data=[eval(line) for line in training_data]
print(len(training_data))

# test_filename1 = './annotated/user_queries_train_07_25_23_preprocessed_unique_Values.txt'

# with open (test_filename1, 'r',encoding="utf-8") as file:  
#     test_data1=[line.strip() for line in file]


test_filename2 = './annotated/user_queries_test.txt'

with open (test_filename2, 'r',encoding="utf-8") as file:  
    test_data2=[line.strip() for line in file]

test_data  = test_data2 
test_data = list(set(test_data))
test_data=[eval(line) for line in test_data]
#test_data = random.sample(test_data, 70000)
print(len(test_data))

618672
50700


In [15]:
training_data+=labeled_data_from_survey
len(training_data)

653802

In [16]:
test_data+= labeled_test_data
len(test_data)

56910

In [17]:
model = "en_core_web_lg"
iterations=1
# Load an existing model or create a new one
if model is not None:
    nlp = spacy.load(model)
    print(f"Loaded model: {model}")
else:
    nlp = spacy.blank("en")
    print("Created blank 'en' model")


Loaded model: en_core_web_lg


In [18]:
db = DocBin() # create a DocBin object
reject_count = 0
for text, annot in tqdm(training_data): # data in previous format
    doc = nlp.make_doc(text) # create doc object from text
    ents = []
    for start, end, label in annot["entities"]: # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            #print("Skipping entity")
            reject_count+=1
        else:
            ents.append(span)
    doc.ents = ents # label the text with the ents
    db.add(doc)

db.to_disk("./train.spacy") # save the docbin object
print(reject_count)

100%|██████████| 653802/653802 [02:03<00:00, 5295.25it/s]


1284


In [19]:
db = DocBin() # create a DocBin object
reject_count=0
for text, annot in tqdm(test_data): # data in previous format
    doc = nlp.make_doc(text) # create doc object from text
    ents = []
    for start, end, label in annot["entities"]: # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            #print("Skipping entity")
            reject_count+=1
        else:
            ents.append(span)
    doc.ents = ents # label the text with the ents
    db.add(doc)

db.to_disk("./test.spacy") # save the docbin object
print(reject_count)

100%|██████████| 56910/56910 [00:10<00:00, 5606.95it/s]


86


In [20]:
# !python -m spacy init fill-config base_config.cfg config.cfg

In [21]:
%%time
!python -m spacy train config.cfg --output ./output_500K_01_24_24 --paths.train ./train.spacy --paths.dev ./test.spacy --gpu-id 0

2024-01-24 10:03:03.679326: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-24 10:03:04.514091: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
✔ Created output directory: output_500K_01_24_24
ℹ Saving to output directory: output_500K_01_24_24
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     46.33    0.00    0.00    0.00    0.00
  0  

In [ ]:
nlp1 = spacy.load(r"./output_500K_01_24_24/model-best") #load the best model

In [19]:
doc = nlp1("looking for a uv resistant grade with cti of 600 volts") # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [23]:
doc = nlp1("what i want is a celanyl product with very high density for car windshield use case, i prefer ppe grade") # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [24]:
doc = nlp1("i am looking for a gf30 pps") # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [25]:
doc = nlp1("i am looking for a 10gf pps") # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [26]:
doc = nlp1("Chemical resistant GUR grade with MW 600000 with avg particle size 120 for automotive application".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [27]:
doc = nlp1( "tensile modulus = 3000 MPa +/-10%".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [49]:
doc = nlp1( "gf / min filled, density of min 1000".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [29]:
doc = nlp1( "what can replace plastron PAX - CF40 - 02".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [30]:
doc = nlp1( "material similar to offset123".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [31]:
doc = nlp1( "offset abasf23ab".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [32]:
doc = nlp1( "offset to donlad - 431ck,  for bumper application".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [33]:
doc = nlp1( "offset grade to eco-r".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [35]:
doc = nlp1( "eco r".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [ ]:
doc = nlp1( "eco r".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [36]:
doc = nlp1( "lower than 10% gf/min, density less than 200, car application".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [37]:
doc = nlp1( "lower than 10% gf, density less than 200, good mechanical properties, car application".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [38]:
doc = nlp1( "Mineral filled LCP with UL yellow card at 0.4mm thickness and flame class rating of V0".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [39]:
doc = nlp1( "offset PPC grade PPS 1140A6 - HD9100 for fuel capless part".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [41]:
doc = nlp1( "bumper, uv resistance".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [42]:
doc = nlp1( "uv resistance, bumper".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [43]:
doc = nlp1( "cti 120".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [48]:
doc = nlp1( "hai 20".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [50]:
doc = nlp1( "flame rating of v0 at 0.3mm".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [54]:
doc = nlp1( "flame rating of v0 at thickness of 0.3mm".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [57]:
doc = nlp1( "flame rating of v0, thickness of 0.3mm".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [61]:
doc = nlp1( "thermalcondutivity grade".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [63]:
doc = nlp1( "flameretardent grade".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [64]:
doc = nlp1( "ptfe".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [7]:
doc = nlp1( "rohs".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [6]:
doc = nlp1( "rohs 2011 complaint".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [8]:
doc = nlp1( "dimensional change".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [ ]:
doc = nlp1( "dimensional change".lower()) # input sample text
spacy.displacy.render(doc, style="ent", jupyter=True)

In [32]:
import json
import ast
import pandas as pd

# filename = './annotated/user_queries_test_22_06_23.txt'

# with open (filename, 'r',encoding="utf-8") as file:  
#     testing_data=[line.strip() for line in file]
# print(len(testing_data) ) 
# testing_data = list(set(testing_data))
# testing_data=[eval(line) for line in testing_data]
# print(len(testing_data) ) 

In [44]:
def extract_entities(text,ner_model):
    doc = ner_model(text.lower())
    result = []
    for ent in doc.ents:
        result.append((ent.start_char,ent.end_char,ent.label_,ent.text))
    return result

In [45]:
with open ('./annotated/survey_query_test_data.txt', 'r') as file:  
    annotated_test_data=[eval(line.strip()) for line in file]

random.shuffle(annotated_test_data)

user_queries_prediction = []
for query in annotated_test_data:
    result = extract_entities(query[0],nlp1)
    user_queries_prediction.append((query[0], {'entities': result}))

In [46]:
with open ('./annotated/model_survey_result_prediction.txt', 'w', encoding="utf-8") as file:  
    for line in user_queries_prediction:  
        file.write(str(line))  
        file.write('\n')

In [63]:
from spacy.scorer import Scorer
from spacy.training import Example

def evaluate(ner_model, samples):
    scorer = Scorer(ner_model)
    example = []
    for sample in samples:
        pred = ner_model(sample[0])
        #print(pred, sample[1]['entities'])
        temp_ex = Example.from_dict(pred, {'entities': sample[1]['entities']})
        example.append(temp_ex)
    scores = scorer.score(example)
    
    return scores

In [ ]:
evaluate(nlp1, testing_data)